# Vetch — Inference Waste Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prismatic-labs/vetch/blob/main/demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/vetch.svg)](https://pypi.org/project/vetch/)

**Vetch detects stalled agents, RAG bloat, and runaway inference — then warns, kills, or reroutes wasteful calls before they burn budget.**

Metadata-only. No prompts or completions read. Fail-open always.

---

### What counts as inference waste?

| Pattern | What it looks like |
|---------|-------------------|
| **STALL-001** | Agent iterating without meaningful output progress |
| **CACHE-001** | Repeated identical input structure that could be cached |
| **RAG-001** | Retrieval context overwhelming the prompt (high input:output ratio) |
| **BABBLE-001** | Unusually long outputs regardless of task complexity |

---

**This notebook is runnable end-to-end with no API keys.** The simulation cells use `ctx.capture()` — the same hook every provider interceptor calls — to generate realistic waste patterns without hitting an LLM API.

In [ ]:
!pip install -q vetch

## 1 · CLI demos (no API key needed)

These work immediately after install.

In [ ]:
# Estimate energy, carbon, and cost for any model — no call made
!vetch estimate --model gpt-4o --input-tokens 1000 --output-tokens 500 --region us-east-1

In [ ]:
# Compare models side-by-side
!vetch compare --models gpt-4o,gpt-4o-mini,claude-3-5-sonnet --tokens 1000

In [ ]:
!vetch status

## 2 · The workflow: instrument → observe → audit → act

```
import vetch
vetch.instrument(region="us-east-1", tags={"service": "chat-api"})

# Your existing LLM calls — unchanged.
# Every call is now tracked: cost, energy, carbon, waste signals.
```

After a few days of traffic:

```bash
vetch audit          # last 7 days of stored metadata
vetch audit --window 24h --format json
```

The rest of this notebook simulates that flow without API keys.

## 3 · Simulate waste patterns (no API key)

We use `ctx.capture()` — the hook every provider interceptor calls after extracting usage from a real response — to inject realistic patterns into storage.

In [ ]:
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import vetch
from vetch.context import get_active_context
from vetch.storage import configure_storage, flush_storage

# Use a temp DB for this demo
_tmpdir = tempfile.mkdtemp(prefix="vetch-demo-")
configure_storage(enabled=True, path=Path(_tmpdir) / "demo.db")
print(f"Storage: {_tmpdir}/demo.db")

In [ ]:
# Simulate a stalled agent loop: 20 calls with near-zero output
# This is what STALL-001 looks for — high input similarity, low output

for i in range(20):
    with vetch.wrap(
        region="us-east-1",
        tags={"feature": "agent-loop", "customer": "acme", "env": "prod"},
    ):
        ctx = get_active_context()
        ctx.capture(
            model="gpt-4o",
            provider="openai",
            usage={"text": {"input_tokens": 500, "output_tokens": 2}},
            accumulated_chars=2,
        )

print("Stall pattern: 20 calls, 500 input tokens, 2 output tokens each")

In [ ]:
# Simulate RAG bloat: retrieval context dwarfing the actual answer

for i in range(15):
    with vetch.wrap(
        region="us-east-1",
        tags={"feature": "rag-search", "customer": "beta", "env": "prod"},
    ):
        ctx = get_active_context()
        ctx.capture(
            model="gpt-4o",
            provider="openai",
            usage={"text": {"input_tokens": 8000, "output_tokens": 35}},
            accumulated_chars=35,
        )

print("RAG bloat pattern: 15 calls, 8000 input tokens, 35 output tokens each")
print(f"Average input:output ratio: {8000/35:.0f}:1")

In [ ]:
# Simulate excessive generation: model producing very long outputs

for i in range(12):
    with vetch.wrap(
        region="us-east-1",
        tags={"feature": "summarizer", "env": "prod"},
    ):
        ctx = get_active_context()
        ctx.capture(
            model="gpt-4o",
            provider="openai",
            usage={"text": {"input_tokens": 300, "output_tokens": 2200}},
            accumulated_chars=2200,
        )

flush_storage(timeout=10.0)
print("Babble pattern: 12 calls, avg 2,200 output tokens each")
print("All 47 simulated events flushed to storage.")

## 4 · Run the audit

In [ ]:
from datetime import timedelta
from vetch.audit_report import build_audit_report, format_audit_report

now = datetime.now(timezone.utc)
report = build_audit_report(
    start=now - timedelta(hours=1),
    end=now + timedelta(minutes=5),
)

print(f"Total requests:       {report.total_requests}")
print(f"Total tokens:         {report.total_tokens:,}")
print(f"Total cost:           ${report.total_cost_usd:.4f}")
print(f"Observed avoidable:   ${report.observed_avoidable_cost_usd:.4f}")
print(f"Projected monthly:    ${report.projected_monthly_avoidable_cost_usd:.2f}")
print(f"Tagged fraction:      {report.data_quality.tagged_fraction:.0%}")
print()
print(f"Findings ({len(report.findings)}):")
for f in report.findings:
    print(f"  [{f.severity}] {f.code} — {f.title}")
    print(f"    Scope: {f.scope}")
    print(f"    Confidence: {f.confidence}")
    print(f"    Action: {f.recommended_action[:80]}")
    print()

In [ ]:
# Full markdown report (same as `vetch audit --format markdown`)
from IPython.display import Markdown, display

display(Markdown(format_audit_report(report, "markdown")))

In [ ]:
# Machine-readable JSON — useful for CI or downstream processing
import json

data = json.loads(format_audit_report(report, "json"))
print(json.dumps({
    "total_requests": data["total_requests"],
    "total_cost_usd": data["total_cost_usd"],
    "observed_avoidable_cost_usd": data["observed_avoidable_cost_usd"],
    "findings": [{"code": f["code"], "severity": f["severity"], "confidence": f["confidence"]}
                 for f in data["findings"]],
}, indent=2))

## 5 · Attribution — which feature, customer, and team?

Tags accumulate in storage so `vetch audit --tags customer=acme` shows only that customer's waste patterns.

In [ ]:
from vetch.storage import query_usage

print("By feature:")
for feature in ("agent-loop", "rag-search", "summarizer"):
    s = query_usage(
        start=now - timedelta(hours=1),
        end=now + timedelta(minutes=5),
        tags={"feature": feature},
    )
    print(f"  {feature:<15} {s.total_requests:>3} requests  "
          f"${s.total_cost_usd:.4f}  "
          f"{s.total_input_tokens:>7,} input tokens")

In [ ]:
# Tag breakdowns from the audit report
print("Breakdown rows in report:")
for row in report.breakdowns[:10]:
    print(f"  {row.dimension:<12} {row.value:<15} "
          f"{row.requests:>3} req  ${row.cost_usd:.4f}")

## 6 · Act on findings — warn, kill, reroute

Once you've validated advisories in warn-only mode, promote to automatic action.

In [ ]:
import vetch
from vetch.stats import SessionStats
from vetch.advisory import generate_advisories

# warn — log on next stalled call, inference continues
vetch.set_stall_action("warn")

# kill — raise StallDetected on next stalled call
# vetch.set_stall_action("kill")

# reroute — silently substitute a cheaper model
# vetch.set_stall_action("reroute", fallback_model="gpt-4o-mini")

print("Stall action set to: warn")
print()

# Advisory confidence scales with evidence
stats_low  = SessionStats()
stats_high = SessionStats()

for _ in range(10):
    stats_low.update({"model": "gpt-4o", "usage": {"text": {"input_tokens": 500, "output_tokens": 2}}})

for _ in range(20):
    stats_high.update({
        "model": "gpt-4o",
        "usage": {"text": {"input_tokens": 500, "output_tokens": 2}},
        "estimated_cost_usd": 0.40,
    })

for label, stats in (("10 calls, no cost data", stats_low),
                      ("20 calls, $8 stalled", stats_high)):
    ads = [a for a in generate_advisories(stats) if a.code == "STALL-001"]
    if ads:
        print(f"{label}: severity={ads[0].severity}, requests={ads[0].request_count}")
    else:
        print(f"{label}: no stall advisory")

## 7 · With a real API key (optional)

Uncomment one section. `vetch.instrument()` patches the SDK automatically — your existing code is unchanged.

In [ ]:
# --- OpenAI ---
# !pip install -q openai
#
# import os, vetch
# from openai import OpenAI
#
# os.environ["OPENAI_API_KEY"] = "sk-..."   # your key
# vetch.instrument(region="us-east-1", tags={"feature": "demo"})
# vetch.set_stall_action("warn")
#
# client = OpenAI()
# with vetch.wrap() as ctx:
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": "Say hello in 5 words"}],
#     )
#     print(response.choices[0].message.content)
#
# print(f"Cost:    ${ctx.event['estimated_cost_usd']:.6f}")
# print(f"Energy:  {ctx.event['estimated_energy_wh']:.6f} Wh")
# print(f"Carbon:  {ctx.event['estimated_carbon_g']:.6f} gCO2e")
# vetch.uninstrument()

print("Uncomment the block above and add your key to run a real call.")

In [ ]:
# --- Anthropic ---
# !pip install -q anthropic
#
# import os, vetch
# import anthropic
#
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # your key
# vetch.instrument(region="us-east-1", tags={"feature": "demo"})
#
# client = anthropic.Anthropic()
# with vetch.wrap() as ctx:
#     response = client.messages.create(
#         model="claude-3-5-haiku-latest",
#         max_tokens=50,
#         messages=[{"role": "user", "content": "Say hello in 5 words"}],
#     )
#     print(response.content[0].text)
#
# print(f"Cost:   ${ctx.event['estimated_cost_usd']:.6f}")
# print(f"Energy: {ctx.event['estimated_energy_wh']:.6f} Wh")
# print(f"Carbon: {ctx.event['estimated_carbon_g']:.6f} gCO2e")
# print(f"Cache read tokens: {ctx.event.get('cache_read_tokens', 0)}")
# vetch.uninstrument()

print("Uncomment the block above and add your key to run a real call.")

## 8 · OTLP export — advisories in Grafana / Datadog

v0.5.0 adds advisory events to OTLP. Configure once and `vetch.advisories_fired_total{code="STALL-001"}` shows up in your existing dashboards.

In [ ]:
# !pip install -q opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc
#
# import vetch
#
# # Honeycomb
# vetch.configure_otlp_export(
#     endpoint="https://api.honeycomb.io",
#     headers={"x-honeycomb-team": "your-api-key"},
# )
#
# # Grafana Cloud
# vetch.configure_otlp_export(
#     endpoint="https://otlp-gateway-prod-us-central-0.grafana.net/otlp",
#     headers={"Authorization": "Basic base64-credentials"},
# )
#
# # Export an advisory manually (e.g. from a batch audit)
# vetch.export_advisory_otlp(
#     "STALL-001", "CRITICAL", "kill",
#     session_id="sess-abc",
#     model="gpt-4o",
#     estimated_waste_usd=8.50,
#     tags={"feature": "agent-loop", "customer": "acme"},
# )

print("Metrics exported to OTLP backends:")
print("  vetch.requests_total          — counter, by model/provider/region")
print("  vetch.energy_wh               — histogram, per call")
print("  vetch.carbon_g                — histogram, per call")
print("  vetch.cost_usd                — histogram, per call")
print("  vetch.advisories_fired_total  — counter, by code/severity/action  (v0.5.0)")
print("  vetch.advisory_waste_usd      — histogram, estimated waste per advisory  (v0.5.0)")

## 9 · Energy and carbon methodology

Every call includes `estimated_energy_wh`, `estimated_carbon_g`, and `energy_tier`. Tiers reflect estimate confidence — use them before acting on numbers.

| Tier | Name | Uncertainty | Source |
|------|------|-------------|--------|
| 0 | Measured | ±10–20% | Direct GPU measurement (pynvml) |
| 1 | Vendor-Published | ±20–50% | Official provider benchmark data |
| 2 | Validated | ±50–100% | Crowdsourced aggregates |
| 3 | Estimated | Order of magnitude | Parameter-based calculation |

Most cloud models are Tier 1 (GPT-4o, Claude 3.7, DeepSeek-R1) or Tier 3. Run `vetch methodology` for full provenance.

In [ ]:
!vetch methodology | head -60

## Next steps

```bash
pip install vetch
```

```python
import vetch
vetch.instrument(region="us-east-1", tags={"service": "my-service"})
# ... your existing LLM calls ...
```

```bash
vetch audit          # after a week of traffic
```

- **[QUICKSTART.md](https://github.com/prismatic-labs/vetch/blob/main/QUICKSTART.md)** — 60-second setup
- **[README](https://github.com/prismatic-labs/vetch)** — full SDK docs
- **[Issues](https://github.com/prismatic-labs/vetch/issues)** — bugs and feature requests
- **[prismaticlabs.ai](https://prismaticlabs.ai)** — paid audit reviews

In [ ]:
# Clean up demo storage
from vetch.storage import configure_storage, shutdown_storage
import shutil

shutdown_storage()
configure_storage(enabled=False)
shutil.rmtree(_tmpdir, ignore_errors=True)
print("Demo storage cleaned up.")